In [2]:
from pathlib import Path
import sys

# Google Colab provides this module; a normal local Python environment does not.
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/NeuroVisionAI')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
import zipfile
import tempfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from introduction_to_deeplearning.config import CLASS_NAMES
from introduction_to_deeplearning.data import load_images, save_splits

# This same Drive folder will be read by training.ipynb.
OUTPUT_PATH = ROOT / 'data' / 'processed'
DRIVE_DATA_FOLDER = None  # Example: Path('/content/drive/MyDrive/Data')

if DRIVE_DATA_FOLDER is not None:
    DATA_PATH = Path(DRIVE_DATA_FOLDER)
else:
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f'Dataset archive not found: {ZIP_PATH}. Check its Drive path.')
    # Read one archive from Drive, then work with small images on Colab's disk.
    # A fresh temporary directory avoids mixing files from two dataset versions.
    EXTRACT_DIR = Path(tempfile.mkdtemp(prefix='neurovision_', dir='/content'))
    local_zip = EXTRACT_DIR / 'dataset.zip'
    shutil.copy2(ZIP_PATH, local_zip)
    with zipfile.ZipFile(local_zip) as archive:
        archive.extractall(EXTRACT_DIR / 'images')

        # Accept class folders directly in the archive or inside a folder such as Data.
        candidates = []
        for glioma_folder in (EXTRACT_DIR / 'images').rglob('glioma'):
            parent = glioma_folder.parent
            if all((parent / name).is_dir() for name in CLASS_NAMES):
                candidates.append(parent)
        if len(candidates) != 1:
            raise ValueError(
                'Expected exactly one folder containing glioma, meningioma, notumor, pituitary. '
                f'Found {len(candidates)}. Inspect the archive structure first.'
            )
        DATA_PATH = candidates[0]
else:
    DATA_PATH = ROOT / 'data' / 'raw'

print('Reading images from:', DATA_PATH)
print('Saving fixed splits to:', OUTPUT_PATH)


ModuleNotFoundError: No module named 'introduction_to_deeplearning'

In [ ]:
images, labels, records, rejected = load_images(DATA_PATH)
print('Images:', images.shape, images.dtype)
print('Rejected files:', len(rejected))
display(rejected.head(10))
display(records.head())


## 3. Inspect the class balance and sample images


In [ ]:
counts = np.bincount(labels, minlength=len(CLASS_NAMES))
plt.bar(CLASS_NAMES, counts)
plt.ylabel('Images')
plt.title('Class balance after removing exact duplicates')
plt.show()
print(dict(zip(CLASS_NAMES, counts.tolist())))

fig, axes = plt.subplots(4, 3, figsize=(10, 11))
for class_index, name in enumerate(CLASS_NAMES):
    selected = np.flatnonzero(labels == class_index)[:3]
    for column, ax in enumerate(axes[class_index]):
        ax.axis('off')
        if column < len(selected):
            ax.imshow(images[selected[column], ..., 0], cmap='gray', vmin=0, vmax=255)
            ax.set_title(name)
plt.tight_layout()
plt.show()


## 4. Understand normalization
Dividing by 255 changes the numerical scale from [0, 255] to [0, 1]. The two
pictures should look identical when each display uses the correct scale.
This example does not change `images` or normalize the saved arrays.


In [ ]:
before = images[0, ..., 0]
after = before.astype(np.float32) / 255.0
print('Before:', before.dtype, before.min(), before.max())
print('After:', after.dtype, after.min(), after.max())
fig, axes = plt.subplots(1, 2)
axes[0].imshow(before, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original uint8')
axes[1].imshow(after, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Normalized float32')
plt.show()


## 5. Create and save fixed splits
Use 70% training, 15% validation and 15% test, stratified to preserve class proportions.
We split indices so that image, label and source path always stay together.
A manifest CSV preserves paths and hashes for each split. One-hot labels match our
four-class softmax and categorical cross-entropy loss.

This cell **replaces the processed arrays** at OUTPUT_PATH in Drive. After this preparation,
use only training and validation while choosing the model. Test stays untouched.


In [ ]:
save_splits(images, labels, records, OUTPUT_PATH)
rejected.to_csv(OUTPUT_PATH / 'rejected_files.csv', index=False)
print('Next: open training.ipynb. Use this same processed folder for every experiment.')


## 6. What to write in your report
Record where the dataset came from and its license, rejected and duplicate counts,
class balance, split sizes, and the missing patient IDs. The paths in manifests are
local references; do not publish private images or identifying file names.
